# Delta Incremental Ingestion Pipeline

## Overview
This notebook orchestrates the incremental ingestion of Excel files into Delta Lake tables.
It supports both full loads and incremental updates with watermark tracking.

## Prerequisites
- Access to Bronze layer volumes (`{environment}_bronze_prelanding_minsur`)
- Excel files stored in the configured file server path
- Delta table existence for incremental loads (if watermark is configured)

## 1. Library Imports & Environment Setup
Import required libraries for data processing, Excel manipulation, and Delta operations.
Disable Arrow optimization to prevent Spark-Pandas compatibility issues.

In [0]:
%run ./Functions

In [0]:
# Import data processing libraries
import pandas as pd
import numpy as np
import re
import unicodedata
from pyspark.sql import functions as F
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException
from pyspark.sql.functions import col

# Disable Arrow optimization to avoid Spark-Pandas compatibility issues
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# Suppress openpyxl warnings about Excel file formatting that don't affect data reading
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

## 2. Widget Parameter Definition
Define notebook widgets to accept runtime parameters for flexible execution.
These parameters control file paths, table names, Excel ranges, and incremental logic.

In [0]:
# Core identification parameters
dbutils.widgets.text("Id", "", "Id")
dbutils.widgets.text("Environment", "", "Environment")

# Source file configuration
dbutils.widgets.text("ExcelPath", "", "Excel Path")
dbutils.widgets.text("TableName", "", "Table Name")
dbutils.widgets.text("SheetName", "", "Sheet Name")
dbutils.widgets.text("DataRange", "", "Data Range")

# Schema configuration
dbutils.widgets.text("IndexColumns", "", "Index Columns")
dbutils.widgets.text("CustomColumnNames", "", "Custom Column Names")
dbutils.widgets.text("ColumnsToSkip", "", "Columns To Skip")
dbutils.widgets.text("RowsToSkip", "", "Rows To Skip")
dbutils.widgets.text("IsHorizontal", "", "Is Horizontal")
dbutils.widgets.text("MergeNHeaders", "", "Merge N Headers")

# Metadata enrichment parameters
dbutils.widgets.text("ReportDate", "", "Report Date")
dbutils.widgets.text("ForecastName", "", "Forecast Name")

# Incremental load configuration
dbutils.widgets.text("IdColumns", "", "Id Columns")
dbutils.widgets.text("WatermarkColumn", "", "Watermark Column")
dbutils.widgets.text("WatermarkLookBack", "", "Watermark Look Back")
dbutils.widgets.text("WatermarkVal", "", "Watermark Val")
dbutils.widgets.text("Datetime", "", "Datetime")

## 3. Parameter Retrieval & Path Construction
Retrieve widget values and construct dynamic file paths using datetime tokens.
The Excel path supports datetime formatting patterns (e.g., `{yyyyMMddHH}`).

In [0]:
# Retrieve source Excel file configuration from notebook widgets
id_record = dbutils.widgets.get("Id")
excel_file_path = dbutils.widgets.get("ExcelPath")
table_name = dbutils.widgets.get("TableName")
excel_sheet_name = dbutils.widgets.get("SheetName")
excel_cell_range = dbutils.widgets.get("DataRange")
index_columns = dbutils.widgets.get("IndexColumns")
custom_column_names = dbutils.widgets.get("CustomColumnNames")
report_date = dbutils.widgets.get("ReportDate")
forecast_name = dbutils.widgets.get("ForecastName")
skip_col_range = dbutils.widgets.get("ColumnsToSkip")
skip_row_range = dbutils.widgets.get("RowsToSkip")

# Parse boolean and numeric parameters
is_horizontal = (True if str(dbutils.widgets.get("IsHorizontal")).lower() == "true" else False)
header_rows = int(dbutils.widgets.get("MergeNHeaders") or 1)

# Incremental load parameters
id_columns = dbutils.widgets.get("IdColumns")
watermark_column = dbutils.widgets.get("WatermarkColumn")
watermark_look_back = dbutils.widgets.get("WatermarkLookBack")
watermark_val = dbutils.widgets.get("WatermarkVal")
current_datetime = str(dbutils.widgets.get("Datetime") or datetime.now("UTC").strftime("%Y-%m-%d %H:%M:%S"))

environment = dbutils.widgets.get("Environment")

# Apply datetime token replacement to file path (e.g., replace {yyyyMMddHH} with actual datetime)
excel_file_path = excel_file_path.replace(".xlsx", "_{yyyyMMddHH}.xlsx")
final_path = apply_datetime_tokens(excel_file_path, datetime.strptime(current_datetime, "%Y-%m-%d %H:%M:%S"))

# Construct full DBFS path using volume mount points
input_excel_path = f"/Volumes/{environment}_bronze_prelanding_minsur/marcobre_fileserver/marcobre_fileserver/{final_path}"

# Print configuration for debugging and audit purposes
print(f"Record ID: {id_record}")
print(f"Input Path: {input_excel_path}")
print(f"Orientation: {'Horizontal' if is_horizontal else 'Vertical'}")
print(f"Header Rows: {header_rows}")
print(f"Current Watermark: {watermark_val}")

## 4. Excel Range Parsing
Parse Excel range configurations to determine which columns and rows to extract.
Handles skip configurations for both rows and columns.

In [0]:
# Parse the Excel range and skip configurations using custom parsing logic
params = parse_excel_range_advanced(
    table_range=excel_cell_range,
    skip_col_range=skip_col_range,
    skip_row_range=skip_row_range
)

print("Parsed Excel parameters:", params)

## 5. Excel Data Extraction
Read Excel data using pandas with parsed parameters.
Supports both horizontal (transposed) and vertical orientations.
Handles custom column mapping or automatic header construction.

In [0]:
# Read raw Excel data without headers (manual header processing)
df = pd.read_excel(
    input_excel_path,
    sheet_name=excel_sheet_name,
    usecols=params['usecols'],
    skiprows=params['skiprows'],
    nrows=params['nrows'],
    header=None,  # Headers will be processed manually based on configuration
    engine="openpyxl"
)

# Handle horizontal orientation (transposed data)
if is_horizontal:
    # Transpose the DataFrame and rebuild structure
    df_t = df.T.reset_index(drop=True)
    
    # Apply custom column mapping if specified
    if index_columns != "" and custom_column_names != "":
        col_indexes = [int(i) for i in index_columns.split(",")]
        new_column_names = [c.strip() for c in custom_column_names.split(",")]
        
        # Validate column mapping consistency
        if len(col_indexes) != len(new_column_names):
            raise ValueError(
                f"Number of index columns ({len(col_indexes)}) does not match "
                f"number of custom column names ({len(new_column_names)})"
            )
        
        # Select specified columns and apply custom names
        df_final = df_t.iloc[:, col_indexes]
        df_final.columns = new_column_names
    else:
        # Build headers from first N rows and remove them from data
        headers = build_headers(df_t, header_rows)
        df_t.columns = headers
        df_final = df_t.iloc[header_rows:].reset_index(drop=True)
else:
    # Vertical orientation processing
    if index_columns != "" and custom_column_names != "":
        # Custom column selection for vertical orientation
        col_indexes = [int(i) - 1 for i in index_columns.split(",")]
        new_column_names = [c.strip() for c in custom_column_names.split(",")]
        
        # Validate mapping
        if len(col_indexes) != len(new_column_names):
            raise ValueError(
                f"Number of index columns ({len(col_indexes)}) does not match "
                f"number of custom column names ({len(new_column_names)})"
            )
        
        df_final = df.iloc[:, col_indexes]
        df_final.columns = new_column_names
    else:
        # Automatic header construction from first N rows
        headers = build_headers(df, header_rows)
        df.columns = headers
        # Remove header rows from actual data
        df = df.iloc[header_rows:].reset_index(drop=True)
        df_final = df.copy()

## 6. Metadata Enrichment
Extract additional metadata from specified Excel cells (Report Date and Forecast Name).
Append these as new columns to the dataset for traceability.

In [0]:
# Initialize metadata variables
report_date_val = ""
forecast_name_val = ""

# Extract Report Date from specified cell if configured
if report_date.strip():
    report_params = parse_excel_range_advanced(f"{report_date.strip()}:{report_date.strip()}")
    report_df = pd.read_excel(
        input_excel_path,
        sheet_name=excel_sheet_name,
        usecols=report_params['usecols'],
        skiprows=report_params['skiprows'],
        nrows=report_params['nrows'],
        header=None,
        engine="openpyxl"
    )
    report_date_val = str(report_df.iloc[0, 0])
    # Add report date column to final dataset
    df_final = df_final.copy()
    df_final.loc[:, "report_date"] = report_date_val

# Extract Forecast Name from specified cell if configured
if forecast_name.strip():
    forecast_params = parse_excel_range_advanced(f"{forecast_name.strip()}:{forecast_name.strip()}")
    forecast_df = pd.read_excel(
        input_excel_path,
        sheet_name=excel_sheet_name,
        usecols=forecast_params['usecols'],
        skiprows=forecast_params['skiprows'],
        nrows=forecast_params['nrows'],
        header=None,
        engine="openpyxl"
    )
    forecast_name_val = str(forecast_df.iloc[0, 0])
    # Add forecast name column to final dataset
    df_final = df_final.copy()
    df_final.loc[:, "forecast"] = forecast_name_val

## 7. Data Cleansing & Standardization
Apply data quality transformations:
- Sanitize column names (remove special characters, normalize)
- Handle blank/unnamed columns
- Resolve duplicate column names
- Normalize data types
- Remove empty rows

In [0]:
# Store original column names for transformation tracking
original_columns = df_final.columns.tolist()

# Sanitize column names to SQL-friendly format
df_final.columns = [sanitize_column(c) for c in df_final.columns]

# Fill blank columns with standardized unnamed names (e.g., unnamed_1, unnamed_2)
df_final = fill_blank_columns(df_final)

# Resolve duplicate column names by appending numeric suffixes
df_final.columns = fix_duplicate_columns(df_final.columns)

# Normalize data types (convert strings to numeric where possible, standardize dates)
df_final = normalize_dataframe_types(df_final)

# Remove rows where all values are NaN (completely empty rows)
df_final = drop_all_nan_rows(df_final)

# Construct forecast suffix for table naming if forecast name exists
final_forecast_name = ""
if forecast_name_val:
    final_forecast_name = f"_{forecast_name_val.lower().replace(' ', '_')}"

# Display column name transformations for audit trail
print("Column mapping (Original -> Sanitized):")
for original, sanitized in zip(original_columns, df_final.columns):
    print(f"  {original}  -->  {sanitized}")

## 8. Spark DataFrame Conversion
Convert processed pandas DataFrame to Spark DataFrame for Delta operations.
Display schema and sample data for validation.

In [0]:
print(f"Forecast suffix: {final_forecast_name}")

# Convert to Spark DataFrame for Delta Lake operations
df_table = spark.createDataFrame(df_final)

# Display sample data for validation
display(df_table)

## 9. Delta Table Operations
Determine load type based on watermark configuration:
- **Full Load**: No watermark configured (overwrite mode)
- **Incremental Load**: Watermark configured (merge/upsert mode)

For incremental loads:
- Calculate maximum watermark value
- Apply lookback filter if configured
- Execute Delta Merge (UPSERT) operation using id_columns as key

In [0]:
# Normalize Excel sheet name for table naming (remove special characters)
final_excel_sheet = excel_sheet_name.lower().replace(' ', '_').replace('.', '').replace('/', '_')

# Normalize destination table name
final_destination_table = f"{slugify(table_name.lower().replace(' ', '_').replace('.', '').replace('/', '_'))}"

# Combine table name with forecast suffix
new_table_name = f"{final_destination_table}{final_forecast_name}"

# Determine load strategy based on watermark configuration
if watermark_val.strip() == "" and watermark_look_back.strip() == "":
    # FULL LOAD: No watermark configured - overwrite existing table
    df_table.write \
        .format("delta") \
        .option("mergeSchema", "true") \
        .mode("overwrite") \
        .saveAsTable(f"{environment}_bronze_landing_minsur.marcobre_fileserver.{new_table_name}")

    print(f"Full load completed. Delta table written to: {environment}_bronze_landing_minsur.marcobre_fileserver.{new_table_name}")
else:
    # INCREMENTAL LOAD: Watermark configured - perform merge operation
    print(f"Incremental load configuration:")
    print(f"  ID Columns: {id_columns}")
    print(f"  Watermark Column: {watermark_column}")
    print(f"  Lookback Period: {watermark_look_back}")
    print(f"  Previous Watermark: {watermark_val}")
    
    # Calculate maximum watermark value from incoming data
    try:
        max_incremental_value = (
            df_table
                .select(F.max(F.col(watermark_column)).alias("Result"))
                .first()["Result"]
        )
    except Exception:
        max_incremental_value = None

    # Apply lookback filter if configured (e.g., "3d" for 3 days lookback)
    if watermark_look_back.strip():
        print(f"Original row count: {df_table.count()}")
        
        # Adjust watermark value by lookback period
        new_filter_value = adjust_value(max_incremental_value, watermark_look_back.strip())
        
        # Filter dataset to only include records after lookback date
        df_table = df_table.filter(
            col(watermark_column) > new_filter_value
        )
        print(f"Filtered row count (after lookback): {df_table.count()}")

    # Validate that ID columns are specified for merge key
    if id_columns == "":
        raise ValueError("Id columns must be specified for incremental load (merge operation)")
    
    # Reference existing Delta table for merge operation
    oldDeltaTable = DeltaTable.forName(
        spark,
        f"{environment}_bronze_landing_minsur.marcobre_fileserver.{new_table_name}"
    )
    
    # Verify target table has data
    count_rows_updates = oldDeltaTable.toDF().count()

    if count_rows_updates > 0:
        # Prepare column mappings for merge operation
        # Insert map: all columns from source
        insert_merge_columns = {
            col: f"newData.{col}" for col in df_table.columns
        }

        # Update map: all columns except ID columns (to preserve key stability)
        update_merge_columns = {
            col: f"newData.{col}"
            for col in df_table.columns
            if col not in id_columns.split(",")
        }

        # Construct merge condition using ID columns (AND combination)
        merge_condition = " AND ".join(
            [f"oldData.{col.strip()} = newData.{col.strip()}"
            for col in id_columns.split(",")]
        )

        # Debug output for merge configuration
        print(f"Merge condition: {merge_condition}")
        print(f"Records to merge: {df_table.count()}")

        # Execute Delta Merge operation (UPSERT)
        (
            oldDeltaTable
                .alias("oldData")
                .merge(
                    source=df_table.alias("newData"),
                    condition=merge_condition
                )
                .whenMatchedUpdate(set=update_merge_columns)      # Update existing records
                .whenNotMatchedInsert(values=insert_merge_columns) # Insert new records
                .execute()
        )

        print(f"Incremental load completed. Max watermark value: {max_incremental_value}")

## 10. Watermark Update
Update the watermark value in the configuration database to enable future incremental loads.
Uses JDBC connection to SQL Server configuration database.

In [0]:
# Retrieve JDBC connection string from Databricks secrets
full_jdbc_url = dbutils.secrets.get(scope="kv-velman-dev", key='velkpi-sqlconnectionstr-jdbc')

# Construct SQL update statement for watermark tracking
sql_stmt = f"""
UPDATE [velkpi].[ExcelConfiguration]
SET WatermarkVal = '{max_incremental_value}'
WHERE Id = '{id_record}'
"""

# Execute SQL update within managed connection context
with jdbc_connection(full_jdbc_url) as conn:
    stmt = conn.createStatement()
    try:
        stmt.execute(sql_stmt)
        print(f"Watermark updated successfully for record ID: {id_record}")
    finally:
        stmt.close()

## 11. Data Verification
Display recent records from the target Delta table to verify successful ingestion.
Results are ordered by watermark column descending to show latest data first.

In [0]:
# Query and display final results for verification
verification_query = f"""
SELECT * 
FROM dev_bronze_landing_minsur.marcobre_fileserver.{new_table_name}
ORDER BY {watermark_column} DESC
"""

spark.sql(verification_query).display()